# Phase 4 — Retrieval evaluation (the real one)

Similarity score is not a correctness measure. A query can score 0.65 against a
generic surah header and be completely wrong, while scoring 0.56 against the
correct verse. That is exactly what is happening in this project, which is why
nobody could tell whether the v2 retrain helped.

This notebook measures **relevance** instead, against ground truth:

- `ayatec_records.json` — 207 Arabic questions, each with gold `verse_keys`
- Metrics: Recall@k, MRR, NDCG@10, Hit Rate@k
- v1 vs v2, with a paired significance test
- An ablation over two corpus defects we identified

Runtime → T4 GPU → Run all.


## 1. Setup

In [ ]:
import torch, os, shutil, subprocess
print("CUDA:", torch.cuda.is_available())
from google.colab import drive; drive.mount('/content/drive')

ROOT  = "/content/drive/MyDrive"
P1    = f"{ROOT}/Phase1_Project/MemberB_B4_B6_output"
P1FIX = f"{ROOT}/Phase1_Project/data_fix_output"
GV2   = f"{ROOT}/Phase3_Project/guardrail_output_v2"

PROJECT="/content/QuranicRAG"
shutil.rmtree(PROJECT, ignore_errors=True)
os.makedirs(f"{PROJECT}/src", exist_ok=True); os.chdir(PROJECT)
shutil.rmtree("/content/_repo", ignore_errors=True)
subprocess.run(["git","clone","--depth","1",
  "https://github.com/Laiba-Noor/quranic-rag-hallucination-free.git","/content/_repo"],check=True)
for f in os.listdir("/content/_repo/src"):
    if f.endswith(".py"): shutil.copy(f"/content/_repo/src/{f}", f"src/{f}")
for f in os.listdir(f"{GV2}/src"):
    if f.endswith(".py"): shutil.copy(f"{GV2}/src/{f}", f"src/{f}")
!pip install -q sentence-transformers hnswlib scipy
print("ready")

## 2. Load both systems

In [ ]:
import shutil, time, sys, json
t=time.time()
shutil.copytree(f"{P1}/b5_real_finetuned",     "m_v1", dirs_exist_ok=True)
shutil.copytree(f"{P1}/index",                 "i_v1", dirs_exist_ok=True)
shutil.copytree(f"{GV2}/b5_real_finetuned_v2", "m_v2", dirs_exist_ok=True)
shutil.copytree(f"{GV2}/index_v2",             "i_v2", dirs_exist_ok=True)
print(f"copied in {time.time()-t:.0f}s")

sys.path.insert(0,"src")
from sentence_transformers import SentenceTransformer
from b6_build_index_and_retrieval_api import load_index, RetrievalAPI

def build(mdir, idir):
    m = SentenceTransformer(mdir)
    idx, ent = load_index(dim=m.get_sentence_embedding_dimension(), out_dir=idir)
    return RetrievalAPI(m, idx, ent), ent

api_v1, ent1 = build("m_v1","i_v1")
api_v2, ent2 = build("m_v2","i_v2")
print(len(ent1), len(ent2))

## 3. Ground truth — 207 AyaTEC questions with gold verses

In [ ]:
import json, os, shutil

# ayatec_records.json is NOT on GitHub - it was never committed.
# Look everywhere sensible, then fall back to a manual upload,
# then stash a copy in Drive so this never happens again.
STASH = f"{ROOT}/Phase4_Project/data"
os.makedirs(STASH, exist_ok=True)

CANDIDATES = [
    f"{STASH}/ayatec_records.json",
    f"{ROOT}/Phase2_Project/Roma_output/data/ayatec_records.json",
    f"{P1FIX}/shared_data/ayatec_records.json",
    "/content/_repo/Data/ayatec_records.json",
    "quranNLP/shared/data/ayatec_records.json",
]

path = next((c for c in CANDIDATES if os.path.exists(c)), None)

if path is None:
    print("Not found anywhere. Upload it from your laptop:")
    print("   Research/Colab_Cells/ayatec_records.json\n")
    from google.colab import files
    up = files.upload()
    path = list(up.keys())[0]

# Keep a permanent copy in Drive.
if os.path.abspath(path) != os.path.abspath(f"{STASH}/ayatec_records.json"):
    shutil.copy(path, f"{STASH}/ayatec_records.json")
    print(f"stashed to {STASH}/ayatec_records.json")

with open(f"{STASH}/ayatec_records.json", encoding="utf-8") as f:
    aya = json.load(f)

GOLD = [(r["question"], set(r["verse_keys"])) for r in aya
        if r.get("question") and r.get("verse_keys")]
print(f"\n{len(GOLD)} questions with gold verses")
print("gold-set sizes:", {n: sum(1 for _, g in GOLD if len(g) == n) for n in (1, 2, 3)},
      "| >3:", sum(1 for _, g in GOLD if len(g) > 3))
for q, g in GOLD[:3]:
    print(f"  {q[:55]:<57} -> {sorted(g)}")

## 4. Metrics

Standard IR definitions, binary relevance. Results are ranked by *verse*, so
the duplicate tafsir entries collapse instead of occupying separate slots.


In [ ]:
import math

def ranked_verse_keys(api, query, depth=50):
    """Retrieve deep, then collapse to a ranked list of unique verse_keys."""
    seen, out = set(), []
    for r in api.retrieve(query, top_k=depth):
        vk = r["verse_key"]
        if vk not in seen:
            seen.add(vk); out.append(vk)
    return out

def evaluate(api, gold_pairs, ks=(1,5,10,20), depth=50, keep=None):
    acc = {f"Recall@{k}": [] for k in ks}
    acc.update({f"HitRate@{k}": [] for k in ks})
    acc["MRR"] = []; acc["NDCG@10"] = []
    per_query = []

    for q, gold in gold_pairs:
        ranked = ranked_verse_keys(api, q, depth)
        if keep is not None:
            ranked = [v for v in ranked if keep(v)]

        rr = 0.0
        for i, vk in enumerate(ranked, 1):
            if vk in gold: rr = 1.0/i; break
        acc["MRR"].append(rr)

        dcg = sum(1.0/math.log2(i+1) for i, vk in enumerate(ranked[:10], 1) if vk in gold)
        idcg = sum(1.0/math.log2(i+1) for i in range(1, min(len(gold),10)+1))
        acc["NDCG@10"].append(dcg/idcg if idcg else 0.0)

        for k in ks:
            hits = sum(1 for vk in ranked[:k] if vk in gold)
            acc[f"Recall@{k}"].append(hits/len(gold))
            acc[f"HitRate@{k}"].append(1.0 if hits else 0.0)
        per_query.append(rr)

    return {m: sum(v)/len(v) for m, v in acc.items()}, per_query

print("metrics defined")

## 5. v1 vs v2 — measured on relevance, not similarity

In [ ]:
import time
t=time.time()
res1, rr1 = evaluate(api_v1, GOLD)
res2, rr2 = evaluate(api_v2, GOLD)
print(f"evaluated in {time.time()-t:.0f}s\n")

order = ["Recall@1","Recall@5","Recall@10","Recall@20",
         "HitRate@1","HitRate@5","HitRate@10","HitRate@20","MRR","NDCG@10"]
print(f"{'metric':<14}{'v1':>9}{'v2':>9}{'delta':>10}")
print("-"*42)
for m in order:
    d = res2[m]-res1[m]
    print(f"{m:<14}{res1[m]:>9.4f}{res2[m]:>9.4f}{d:>+10.4f}")

## 6. Is the difference real, or noise?

In [ ]:
from scipy.stats import wilcoxon
import statistics as st

diffs = [b-a for a,b in zip(rr1, rr2)]
nz = [d for d in diffs if d != 0]
print(f"queries where the two differ: {len(nz)}/{len(diffs)}")
print(f"v2 better: {sum(1 for d in nz if d>0)} | v1 better: {sum(1 for d in nz if d<0)}")

if nz:
    stat, p = wilcoxon(rr1, rr2, zero_method="wilcox")
    sd = st.pstdev(diffs) or 1e-9
    print(f"\nWilcoxon signed-rank: p = {p:.4f}")
    print(f"Cohen's d (paired):   {st.mean(diffs)/sd:+.3f}")
    print("VERDICT:", "significant (p<0.05)" if p < 0.05 else "NOT significant - the two are equivalent")

## 7. Ablation — do the corpus defects matter?

Two defects found in `final_cross_reference_index.csv`:

- **630 surah-header entries.** Verse 1 of each surah carries the chapter
  introduction ("تفسير سورة الطور، وهي مكية…") as its tafsir. These are long,
  generic, and match almost any query at ~0.6 — noise magnets.
- **4,502 duplicate texts (36%).** Ibn Kathir comments on blocks of verses;
  the block text was copied onto every verse in it.

Ranking by verse already handles the duplicates. This isolates the headers.


In [ ]:
header_keys = {e["verse_key"] for e in ent2
               if e["source_type"]=="tafsir"
               and e["text"].lstrip().startswith(("تَفْسِيرُ سُورَةِ","تفسير سورة"))}
print(f"{len(header_keys)} verse_keys carry a surah-header tafsir\n")

no_hdr = lambda vk: vk not in header_keys
rows = []
for name, api in [("v1", api_v1), ("v2", api_v2)]:
    base, _ = evaluate(api, GOLD)
    filt, _ = evaluate(api, GOLD, keep=no_hdr)
    rows.append((name, base, filt))

print(f"{'system':<22}{'Recall@10':>11}{'MRR':>9}{'NDCG@10':>10}")
print("-"*52)
for name, base, filt in rows:
    print(f"{name+' (as-is)':<22}{base['Recall@10']:>11.4f}{base['MRR']:>9.4f}{base['NDCG@10']:>10.4f}")
    print(f"{name+' (no headers)':<22}{filt['Recall@10']:>11.4f}{filt['MRR']:>9.4f}{filt['NDCG@10']:>10.4f}")

## 8. Save results to Drive

In [ ]:
import json, os
OUT = f"{ROOT}/Phase4_Project"
os.makedirs(OUT, exist_ok=True)
payload = {
    "n_questions": len(GOLD),
    "v1": res1, "v2": res2,
    "per_query_rr": {"v1": rr1, "v2": rr2},
    "surah_header_verse_keys": len(header_keys),
}
with open(f"{OUT}/phase4_retrieval_eval.json","w") as f:
    json.dump(payload, f, indent=2, ensure_ascii=False)
print("saved:", f"{OUT}/phase4_retrieval_eval.json")

---
## Reading the output

`Recall@10` is the headline: of the gold verses, what fraction appear in the
top 10. `MRR` says how high the first correct verse ranks.

This is the first table of your paper, and the first number in this project
that actually means something. Send Claude sections 5, 6 and 7.
